In [ ]:
import os, torch
print("working folder:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
%cd /content
!git clone https://github.com/samyuktaathreya/pinyin_audio_recognizer_model.git repo
%cd /content/repo/pinyin_scorer
%pip install -q pypinyin

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/pinyin_runs

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo log --oneline -1
!ls *.py
!grep -c "Hiss only" data.py

In [ ]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/ext
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl
!python prepare_folders.py --root /content/ext/standard --out-dir manifests/folders

In [ ]:
!grep -c "Hiss only" data.py
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

In [ ]:
!ps aux | grep "train.py" | grep -v grep | wc -l
!ls /content/repo/pinyin_scorer/manifests /content/ext 2>&1 | head -5
!ls /content/drive/MyDrive/pinyin_runs/stage4/last

In [ ]:
%cd /content/repo/pinyin_scorer

In [ ]:
!setsid nohup python -u train.py --init-from /content/drive/MyDrive/pinyin_runs/stage4/last \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4b \
    --epochs 5 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 100 --augment --workers 2 \
    > /content/train_stage4b.log 2>&1 &

In [ ]:
!tail -3 /content/train_stage4b.log
!grep "^epoch [0-9]* initial" /content/train_stage4b.log

In [ ]:
!python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/stage4b/best \
    --manifests manifests/syl_test.jsonl manifests/folders/test_*.jsonl --out stage4_heldout.csv

In [ ]:
!ps aux | grep "train.py" | grep -v grep | grep -o "\-\-out [^ ]*" | sort -u

In [ ]:
import json
for line in open("/content/drive/MyDrive/pinyin_runs/stage4/log.jsonl"):
    d = json.loads(line)
    print(d["epoch"], *(f"{d[k]:.3f}" for k in
          ["syllable_initial_acc", "syllable_final_acc", "syllable_tone_acc", "syllable_all3_acc"]))

In [ ]:
!pkill -f train.py
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "no-augment-sources" prepare_folders.py; grep -c "duplicate_pairs" find_duplicates.py; grep -c "def speech_power" data.py

In [ ]:
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/std
!ls /content/std/standard

In [ ]:
ROOT = "/content/std/standard"
!python find_duplicates.py {ROOT}/*/

In [ ]:
import pandas as pd
from IPython.display import Audio, display
ex = pd.read_csv("duplicate_examples.csv")
pair = ex[(ex.source_a == "losttheory") & (ex.source_b == "uvm")]
for _, r in pair.sample(3, random_state=0).iterrows():
    print(f"\n{r.syllable}  (similarity {r.similarity})")
    display(Audio(r.file_a)); display(Audio(r.file_b))

In [ ]:
ROOT = "/content/std/standard"
!python prepare_folders.py --root {ROOT} --out-dir manifests/std \
    --test-sources gitcode studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory
!ls manifests/syl_train.jsonl manifests/syl_test.jsonl

In [ ]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl

In [ ]:
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/std/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage5 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

In [ ]:
for name in ["stage2", "stage5"]:
    print(f"\n######## {name}")
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/std/test_*.jsonl --out {name}_std.csv
!cp stage2_std.csv stage5_std.csv /content/drive/MyDrive/

In [ ]:
import json, os, glob, collections
train = [json.loads(l) for f in ["manifests/syl_train.jsonl", "manifests/std/train.jsonl"] for l in open(f)]
per_speaker = collections.Counter(r["speaker"] for r in train)
print(len(train), "training clips:", dict(per_speaker))
train_paths = {os.path.realpath(r["audio"]) for r in train}
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    speakers = {r["speaker"] for r in rows}
    same_files = sum(os.path.realpath(r["audio"]) in train_paths for r in rows)
    print(f"{tf}: {len(rows)} clips, speakers {sorted(speakers)}, "
          f"speakers also in training: {sorted(speakers & set(per_speaker)) or 'none'}, "
          f"identical files in training: {same_files}")

In [ ]:
import re
from pathlib import Path
dst = Path("/content/tp_by_speaker")
for f in Path("/content/tone_perfect").rglob("*.mp3"):
    m = re.match(r"^[a-zü:v]+[1-5]_([A-Za-z0-9]+)", f.name)
    if m:
        d = dst / f"tp_{m.group(1)}"
        d.mkdir(parents=True, exist_ok=True)
        if not (d / f.name).exists():
            (d / f.name).symlink_to(f)
!python find_duplicates.py /content/tp_by_speaker/*/ /content/std/standard/*/ \
    --out tp_dup_examples.csv --summary tp_dup_pairs.csv

In [ ]:
import json, glob, random
from IPython.display import Audio, display
from score import Scorer
scorer = Scorer("/content/drive/MyDrive/pinyin_runs/stage5/best")
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    scored = []
    for r in random.Random(0).sample(rows, min(40, len(rows))):
        res = scorer.score(r["audio"], r["pinyin"])
        scored.append((min(p["score"] for p in res["parts"].values()), r, res["parts"]))
    scored.sort(key=lambda x: x[0])
    print(f"\n===== {tf}")
    for label, group in [("WORST", scored[:3]), ("BEST", scored[-3:])]:
        for conf, r, p in group:
            print(f"{label}: target {r['pinyin']:7} heard {p['initial']['heard']} + {p['final']['heard']} "
                  f"+ tone {p['tone']['heard']}   (weakest part {conf:.2f})")
            display(Audio(r["audio"]))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
# !grep -c "def normalize_loudness" data.py; grep -c "duplicate_pairs" find_duplicates.py

In [ ]:
for name in ["stage2", "stage5"]:
    print(f"\n######## {name}")
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/std/test_*.jsonl --out {name}_norm.csv

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "duplicate_pairs" find_duplicates.py

In [ ]:
import json, os, glob, collections
train = [json.loads(l) for f in ["manifests/syl_train.jsonl", "manifests/std/train.jsonl"] for l in open(f)]
per_speaker = collections.Counter(r["speaker"] for r in train)
print(len(train), "training clips:", dict(per_speaker))
train_paths = {os.path.realpath(r["audio"]) for r in train}
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    speakers = {r["speaker"] for r in rows}
    same_files = sum(os.path.realpath(r["audio"]) in train_paths for r in rows)
    print(f"{tf}: {len(rows)} clips, speakers {sorted(speakers)}, "
          f"speakers also in training: {sorted(speakers & set(per_speaker)) or 'none'}, "
          f"identical files in training: {same_files}")

In [ ]:
import re
from pathlib import Path
dst = Path("/content/tp_by_speaker")
for f in Path("/content/tone_perfect").rglob("*.mp3"):
    m = re.match(r"^[a-zü:v]+[1-5]_([A-Za-z0-9]+)", f.name)
    if m:
        d = dst / f"tp_{m.group(1)}"
        d.mkdir(parents=True, exist_ok=True)
        if not (d / f.name).exists():
            (d / f.name).symlink_to(f)
!python find_duplicates.py /content/tp_by_speaker/*/ /content/std/standard/*/ \
    --out tp_dup_examples.csv --summary tp_dup_pairs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

df = pd.read_csv("stage2_norm.csv")
g = df[df.source == "test_gitcode"]
print(len(g), "gitcode clips the model got wrong\n")
for _, r in g.sample(min(12, len(g)), random_state=1).iterrows():
    wrong = [h for h in ("initial", "final", "tone") if r[f"wrong_{h}"]]
    print(f"target {r.target:8} heard {r.heard_initial} + {r.heard_final} + tone {r.heard_tone}"
          f"   (wrong: {', '.join(wrong)})")
    display(Audio(r.audio))

In [ ]:
%cd /content/repo/pinyin_scorer

In [ ]:
import base64, io, random
import numpy as np, soundfile as sf
from IPython.display import HTML, display
from data import load_audio, SAMPLE_RATE

def clip_tag(path):
    """Audio player for one clip: volume evened out, silence trimmed off the ends."""
    wav = load_audio(path)
    n = len(wav) // 160
    rms = np.sqrt((wav[:n * 160].reshape(n, 160) ** 2).mean(axis=1))
    active = np.where(rms > 0.05 * rms.max())[0]
    if len(active):
        wav = wav[max(0, active[0] * 160 - 3200): (active[-1] + 1) * 160 + 3200]
    buf = io.BytesIO()
    sf.write(buf, wav, SAMPLE_RATE, format="WAV", subtype="PCM_16")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<audio controls preload="none" style="width:110px" src="data:audio/wav;base64,{b64}"></audio>'

def grid(kind="final", speakers=None):
    idx, units = (1, FINALS) if kind == "final" else (0, INITIALS)
    speakers = speakers or sorted(by_speaker)
    rng = random.Random(0)
    head = "<tr><th></th>" + "".join(f"<th>{s}</th>" for s in speakers) + "</tr>"
    rows = []
    for unit in units:
        cells = [f"<th>{unit}</th>"]
        for s in speakers:
            cands = [(p, r) for p, r in by_speaker[s] if p[idx] == unit]
            if not cands:
                cells.append("<td>–</td>")
                continue
            p, r = rng.choice([c for c in cands if c[0][2] == "1"] or cands)
            cells.append(f"<td style='font-size:11px'>{r['pinyin']}<br>{clip_tag(r['audio'])}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f"<div style='overflow-x:auto'><table>{head}{''.join(rows)}</table></div>"))

web = ["davinfifield", "digmandarin", "gitcode", "hotoo", "hugolpz", "mandarintemple",
       "misc", "studycli", "uvm", "yabla", "yoyochinese"]
grid("final", web)

In [ ]:
grid("initial", web)

In [ ]:
checks = [("digmandarin", "lan"), ("digmandarin", "shang"), ("digmandarin", "zhong"),
          ("hugolpz", "nao"), ("hugolpz", "fan"), ("hugolpz", "meng"),
          ("gitcode", "gong"), ("davinfifield", "miao"), ("davinfifield", "weng")]
for spk, base in checks:
    for p, r in by_speaker[spk]:
        if r["pinyin"][:-1] == base:
            print(f"{spk:14} {r['pinyin']:8} {r['audio'].split('/')[-1]}")
            display(Audio(r["audio"]))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!python prepare_folders.py --root /content/std/standard --out-dir manifests/good \
    --test-sources studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory davinfifield digmandarin gitcode hugolpz yoyochinese

In [ ]:
import json, glob, random
import soundfile as sf
from IPython.display import Audio, display

rows = [json.loads(l) for m in ["manifests/good/train.jsonl", *glob.glob("manifests/good/test_*.jsonl")]
        for l in open(m)]
speakers = ["hotoo", "mandarintemple", "misc", "studycli", "uvm", "yabla"]
rng = random.Random()          # different clips every run; use random.Random(0) to repeat a set

for spk in speakers:
    print(f"\n==================== {spk}")
    mine = [r for r in rows if r["speaker"] == spk]
    for tone in "1234":
        pool = [r for r in mine if r["pinyin"].endswith(tone)]
        for r in rng.sample(pool, min(2, len(pool))):
            wav, sr = sf.read(r["audio"])
            print(f"{r['pinyin']:8} {r['audio'].split('/')[-1]}")
            display(Audio(wav.T if wav.ndim > 1 else wav, rate=sr))

In [ ]:
import json, glob, random, base64, io, collections
import soundfile as sf
from IPython.display import HTML, display
from pinyin_units import split_syllable, INITIALS, FINALS
from data import load_audio, SAMPLE_RATE

by_speaker = collections.defaultdict(list)
for m in ["manifests/good/train.jsonl", *glob.glob("manifests/good/test_*.jsonl")]:
    for line in open(m):
        r = json.loads(line)
        parts = split_syllable(r["pinyin"])
        if parts:
            by_speaker[r["speaker"]].append((parts, r))

def clip_tag(path):
    buf = io.BytesIO()
    sf.write(buf, load_audio(path), SAMPLE_RATE, format="WAV", subtype="PCM_16")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<audio controls preload="none" style="width:110px" src="data:audio/wav;base64,{b64}"></audio>'

def grid(kind="initial", speakers=None, seed=None):
    idx, units = (0, INITIALS) if kind == "initial" else (1, FINALS)
    speakers = speakers or sorted(by_speaker)
    rng = random.Random(seed)
    head = "<tr><th></th>" + "".join(f"<th>{s}</th>" for s in speakers) + "</tr>"
    rows = []
    for unit in units:
        cells = [f"<th>{unit}</th>"]
        for s in speakers:
            cands = [r for p, r in by_speaker[s] if p[idx] == unit]
            if not cands:
                cells.append("<td>–</td>")
                continue
            r = rng.choice(cands)
            cells.append(f"<td style='font-size:11px'>{r['pinyin']}<br>{clip_tag(r['audio'])}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f"<div style='overflow-x:auto'><table>{head}{''.join(rows)}</table></div>"))

grid("initial", ["hotoo", "mandarintemple", "misc", "studycli", "uvm", "yabla"])

In [ ]:
!git -C /content/repo pull

In [ ]:
!python dump_probs.py --model /content/drive/MyDrive/pinyin_runs/stage2/best \
    --manifests manifests/syl_test.jsonl manifests/good/train.jsonl manifests/good/test_*.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage2_probs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
SOURCES = ["hotoo", "misc"]
SYLLABLES = ["zhi", "chi", "shi", "ri", "zi", "ci", "si"]

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
df = df[df.source.isin(SOURCES) & df.target.str[:-1].isin(SYLLABLES)].copy()

def heard(row, part):   # the model's top guess for one part, ignoring silence
    cols = [c for c in df.columns if c.startswith(part + ":") and "silence" not in c]
    return max(cols, key=lambda c: float(row[c])).split(":")[1]

for part in ["initial", "final", "tone"]:
    df["heard_" + part] = df.apply(heard, axis=1, part=part)
df["wrong"] = (df.heard_initial != df.target_initial) | (df.heard_final != df.target_final)
df = df.sort_values(["wrong", "source", "target"], ascending=[False, True, True])

print(f"{len(df)} clips, {df.wrong.sum()} where the model heard a different initial or final\n")
for _, r in df.iterrows():
    mark = "WRONG" if r.wrong else "ok   "
    print(f"{mark}  {r.source:6} {r.target:5}  model heard: {r.heard_initial} + {r.heard_final} + tone {r.heard_tone}")
    display(Audio(r.audio))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull

In [ ]:
# Run A: clean data only
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage6a \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 200 --augment --workers 2

In [ ]:
# Run B: the same, plus the two changes
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage6b \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 200 --augment --workers 2 \
    --voice-augment --per-head-mix

In [ ]:
for name in ["stage2", "stage6a", "stage6b"]:
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl --out {name}_good.csv

In [ ]:
!python dump_probs.py --model /content/drive/MyDrive/pinyin_runs/stage6b/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage6b_probs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

CSV = "/content/drive/MyDrive/pinyin_runs/stage6b_probs.csv"
SOURCE = "uvm"       # "uvm", "studycli", "FV3", "MV3", or None for all
PART   = "final"     # "initial", "final" or "tone"
TARGET = "eng"       # what the clip is labelled
HEARD  = "ang"       # what the model heard instead, or None for any wrong answer
LIMIT  = 30          # most clips to play

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
cols = [c for c in df.columns if c.startswith(PART + ":") and "silence" not in c]
probs = df[cols].astype(float)
df["heard"] = probs.idxmax(axis=1).str.split(":").str[1]
df["p_target"] = [2.718281828 ** probs.at[i, f"{PART}:{t}"] for i, t in zip(df.index, df["target_" + PART])]

m = (df["target_" + PART] == TARGET) & (df.heard != TARGET)
if HEARD:  m &= df.heard == HEARD
if SOURCE: m &= df.source == SOURCE
hits = df[m].sort_values("p_target")

print(f"{len(hits)} clips labelled {TARGET} where the model heard {HEARD or 'something else'}\n")
for _, r in hits.head(LIMIT).iterrows():
    print(f"{r.source:9} {r.target:8} heard {r.heard:5} target's probability {r.p_target:.1%}")
    display(Audio(r.audio))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull

In [ ]:
# ---- Session setup: run once at the start of every Colab session (T4 runtime) ----
REPO_URL = "https://github.com/samyuktaathreya/pinyin_audio_recognizer_model.git"   # <- put your repo's address here

from google.colab import drive
drive.mount('/content/drive')

# Code: clone the first time, otherwise pull the latest push
%cd /content
!git clone -q {REPO_URL} repo 2>/dev/null || git -C repo pull -q
%cd /content/repo/pinyin_scorer
%pip install -q pypinyin librosa

# Audio: Colab's disk is wiped every session, so unpack it again
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/std

# Manifests (lists of clips with their labels)
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl
!python prepare_folders.py --root /content/std/standard --out-dir manifests/good \
    --test-sources studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory davinfifield digmandarin gitcode hugolpz yoyochinese

# Checks: every line should end in True, and the GPU line should name a T4
import os, glob, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4")
print("new training code:", "--sentences" in open("train.py").read())
print("threshold rule in score.py:", "DEFAULT_THRESHOLD" in open("score.py").read())
print("Tone Perfect audio:", os.path.exists("/content/tone_perfect"))
print("web audio:", os.path.exists("/content/std/standard/uvm"))
for m in ["manifests/syl_train.jsonl", "manifests/syl_test.jsonl", "manifests/good/train.jsonl"] + sorted(glob.glob("manifests/good/test_*.jsonl")):
    print(f"{m}: {sum(1 for _ in open(m))} clips")

In [ ]:
runs = "/content/drive/MyDrive/pinyin_runs"
candidates = {"probe_w2v_base": "TencentGameMate/chinese-wav2vec2-base",
              "probe_hubert_base": "TencentGameMate/chinese-hubert-base",
              "probe_hubert_large": "TencentGameMate/chinese-hubert-large"}
for name, enc in candidates.items():
    !python train.py --encoder {enc} --freeze-encoder --per-head-mix \
        --train manifests/syl_train.jsonl manifests/good/train.jsonl \
        --dev manifests/syl_test.jsonl --out {runs}/{name} \
        --epochs 10 --head-lr 2e-3 --warmup-steps 100 --workers 2
for name in candidates:
    !python eval_sources.py --model {runs}/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl --out {name}.csv

In [ ]:
# ---- AISHELL setup (needed every session, because Colab's disk is wiped) ----
import os, glob
if not os.path.exists("/content/data_aishell/wav/train"):
    !tar -xzf /content/drive/MyDrive/data_aishell.tgz -C /content
    # AISHELL ships one archive per speaker inside wav/; unpack them, then delete the archives
    !cd /content/data_aishell/wav && for f in *.tar.gz; do tar -xzf "$f" && rm "$f"; done
print("speakers unpacked:", len(glob.glob("/content/data_aishell/wav/*/S*")))

In [ ]:
!python prepare_aishell.py --aishell-dir /content/data_aishell --split train --hours 15 --out manifests/aishell_train.jsonl
!python prepare_aishell.py --aishell-dir /content/data_aishell --split dev --hours 1 --out manifests/aishell_dev.jsonl

# ---- Stage 1 on HuBERT ----
!python train.py --encoder TencentGameMate/chinese-hubert-base \
    --train manifests/aishell_train.jsonl --dev manifests/aishell_dev.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage1_hubert \
    --epochs 11 --per-head-mix --workers 2

In [ ]:
import json
for name in ["stage1/best", "stage1_hubert/best", "stage7_hubert/best"]:
    try:
        cfg = json.load(open(f"/content/drive/MyDrive/pinyin_runs/{name}/encoder/config.json"))
        print(f"{name:20} {cfg['model_type']}")
    except FileNotFoundError:
        print(f"{name:20} not trained yet")

In [ ]:
runs = "/content/drive/MyDrive/pinyin_runs"

# Stage 7: HuBERT, syllables with sentences mixed in
!python train.py --init-from {runs}/stage1_hubert/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --sentences manifests/aishell_train.jsonl --sentence-hours 3 \
    --dev manifests/syl_test.jsonl --out {runs}/stage7_hubert \
    --epochs 8 --lr 2e-5 --head-lr 2e-4 --warmup-steps 200 \
    --augment --voice-augment --per-head-mix --workers 2

# Compare against the earlier models on the unseen voices
for name in ["stage2", "stage6b", "stage7_hubert"]:
    !python eval_sources.py --model {runs}/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl --out {name}_good.csv

# Save every clip's probabilities, for the threshold analysis
!python dump_probs.py --model {runs}/stage7_hubert/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --out {runs}/stage7_hubert_probs.csv

In [ ]:
import pandas as pd, numpy as np, librosa, matplotlib.pyplot as plt

CSV = "/content/drive/MyDrive/pinyin_runs/stage7_hubert_probs.csv"
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
tone_cols = [c for c in df.columns if c.startswith("tone:") and "silence" not in c]
df["heard_tone"] = df[tone_cols].astype(float).idxmax(axis=1).str.split(":").str[1]

groups = {
    "studycli tone 2, heard as 3": (df.source == "studycli") & (df.target_tone == "2") & (df.heard_tone == "3"),
    "studycli tone 2, heard as 2": (df.source == "studycli") & (df.target_tone == "2") & (df.heard_tone == "2"),
    "studycli tone 3":             (df.source == "studycli") & (df.target_tone == "3"),
    "FV3 tone 2 (reference)":      (df.source == "FV3") & (df.target_tone == "2"),
}

def contour(path):
    y, sr = librosa.load(path, sr=16000)
    f0, voiced, _ = librosa.pyin(y, fmin=60, fmax=500, sr=sr, frame_length=1024)
    f0 = f0[voiced]
    return f0, len(f0) * 256 / sr        # pitch values, seconds of voicing

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, (name, mask) in zip(axes, groups.items()):
    clips = df[mask].sample(min(15, mask.sum()), random_state=0)
    curves = [contour(p) for p in clips.audio]
    ref = np.median(np.concatenate([c for c, _ in curves if len(c)]))   # speaker's typical pitch
    for f0, _ in curves:
        if len(f0) > 5:
            ax.plot(np.linspace(0, 1, len(f0)), 12 * np.log2(f0 / ref), alpha=0.6)
    ax.set_title(f"{name}\nvoiced {np.mean([s for _, s in curves]):.2f} s on average")
    ax.set_xlabel("position in syllable"); ax.axhline(0, color="gray", lw=0.5)
axes[0].set_ylabel("semitones from speaker's median")
plt.tight_layout(); plt.show()

In [ ]:
%pip install -q praat-parselmouth

In [ ]:
import numpy as np, pandas as pd, parselmouth

CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
SPEAKERS = ["uvm", "studycli", "hotoo", "mandarintemple", "yabla", "FV3", "MV3"]
PAIRS = [("eng", "ang"), ("en", "an"), ("ou", "ao")]

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)

def vowel_f1(path):
    """F1 (Hz) early in the vowel, before the nasal or second vowel takes over."""
    snd = parselmouth.Sound(path)
    pitch = snd.to_pitch()
    voiced = pitch.xs()[pitch.selected_array["frequency"] > 0]
    if len(voiced) < 5:
        return np.nan
    t0, t1 = voiced[0], voiced[-1]
    formant = snd.to_formant_burg(maximum_formant=5500)
    times = t0 + (t1 - t0) * np.linspace(0.2, 0.4, 5)
    return np.nanmedian([formant.get_value_at_time(1, t) for t in times])

rows = []
for speaker in SPEAKERS:
    for closed, open_ in PAIRS:
        f1 = {}
        for final in (closed, open_):
            clips = df[(df.source == speaker) & (df.target_final == final)].audio
            f1[final] = np.nanmedian([vowel_f1(p) for p in clips])
        rows.append({"speaker": speaker, "pair": f"{closed}/{open_}",
                     "F1 closed": f1[closed], "F1 open": f1[open_],
                     "closed as % of open": 100 * f1[closed] / f1[open_]})
print(pd.DataFrame(rows).round(0).to_string(index=False))

In [ ]:
import numpy as np, pandas as pd
from scipy.signal import resample_poly
from data import load_audio
from score import Scorer

MODEL = "/content/drive/MyDrive/pinyin_runs/stage6b/best"
CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
FACTORS = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10]    # below 1 lowers every frequency, above 1 raises it
GROUPS = [("FV3", "eng"), ("FV3", "ang"), ("MV3", "eng"), ("MV3", "ang"),
          ("yabla", "eng"), ("yabla", "ang")]

scorer = Scorer(MODEL)
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)

def shifted(path, factor):
    """Scale every frequency in the clip (pitch and vowel resonances) by `factor`."""
    wav = load_audio(path)
    return resample_poly(wav, 1000, int(round(1000 * factor))).astype(np.float32)

rows = []
for speaker, final in GROUPS:
    clips = df[(df.source == speaker) & (df.target_final == final)]
    row = {"clips": f"{speaker} {final} ({len(clips)})"}
    for factor in FACTORS:
        heard = [scorer.score(shifted(r.audio, factor), r.target)["parts"]["final"]["heard_unit"]
                 for r in clips.itertuples()]
        row[f"x{factor:.2f}"] = f"{100 * np.mean([h == final for h in heard]):.0f}%"
    rows.append(row)
    print(row, flush=True)

print("\nShare of clips whose final is heard correctly, at each frequency shift:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import numpy as np, pandas as pd
from collections import Counter
from scipy.signal import resample_poly
from data import load_audio
from score import Scorer

MODEL = "/content/drive/MyDrive/pinyin_runs/stage6b/best"
CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
FACTORS = [0.85, 0.90, 1.00]
GROUPS = [("FV3", "ang"), ("FV3", "eng")]

scorer = Scorer(MODEL)
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)

def shifted(path, factor):
    wav = load_audio(path)
    return resample_poly(wav, 1000, int(round(1000 * factor))).astype(np.float32)

for speaker, final in GROUPS:
    clips = df[(df.source == speaker) & (df.target_final == final)]
    for factor in FACTORS:
        heard = Counter(scorer.score(shifted(r.audio, factor), r.target)["parts"]["final"]["heard_unit"]
                        for r in clips.itertuples())
        print(f"{speaker} {final} at x{factor:.2f}: " + ", ".join(f"{u} {n}" for u, n in heard.most_common()))

In [14]:
!python shift_test.py --model {runs}/stage6b/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --sources FV3 MV3 studycli --out {runs}/shift_6b.csv --check

3 clips x 14 conditions
Loading weights: 100% 211/211 [00:00<00:00, 2797.84it/s]
FV3       an1      orig        final heard en    p=0.062  tone heard 1 p=0.974
FV3       an1      praat_1.00  final heard en    p=0.030  tone heard 1 p=0.964
FV3       an1      F0.85       final heard en    p=0.019  tone heard 1 p=0.984
FV3       an1      F0.90       final heard an    p=0.927  tone heard 1 p=0.987
FV3       an1      F1.10       final heard an    p=0.914  tone heard 1 p=0.972
FV3       an1      F1.15       final heard an    p=0.978  tone heard 1 p=0.989
FV3       an1      P-4         final heard an    p=0.944  tone heard 1 p=0.979
FV3       an1      P-2         final heard en    p=0.032  tone heard 1 p=0.965
FV3       an1      P+2         final heard an    p=0.987  tone heard 1 p=0.991
FV3       an1      P+4         final heard an    p=0.749  tone heard 1 p=0.993
FV3       an1      B0.85       final heard an    p=0.408  tone heard 1 p=0.977
FV3       an1      B0.90       final heard en    p

In [15]:
!python shift_test.py --model {runs}/stage6b/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --sources FV3 MV3 studycli --out {runs}/shift_6b.csv

2216 clips x 14 conditions
Loading weights: 100% 211/211 [00:00<00:00, 2730.73it/s]
  100/2216 clips
  200/2216 clips
  300/2216 clips
  400/2216 clips
  500/2216 clips
  600/2216 clips
  700/2216 clips
  800/2216 clips
  900/2216 clips
  1000/2216 clips
  1100/2216 clips
  1200/2216 clips
  1300/2216 clips
  1400/2216 clips
  1500/2216 clips
  1600/2216 clips
  1700/2216 clips
  1800/2216 clips
  1900/2216 clips
  2000/2216 clips
  2100/2216 clips
  2200/2216 clips

=== ALL: final accuracy % (n per cell in brackets)
final        orig praat_1.00      F0.85      F0.90      F1.10      F1.15        P-4        P-2        P+2        P+4      B0.85      B0.90      B1.10      B1.15
an        98 (224)    98 (224)    95 (224)    97 (224)    98 (224)    96 (224)    99 (224)    98 (224)   100 (224)    95 (224)    96 (224)    95 (224)    99 (224)    93 (224)
ang       96 (224)    96 (224)    86 (224)    95 (224)    93 (224)    87 (224)    98 (224)    96 (224)    97 (224)    91 (224)    90 (224)   